In [ ]:
# When running in Google Colab, this cell installs dependencies, clones the
# paper repository, and downloads the required data.  It takes about a minute
# on first run; subsequent cells in the same session are unaffected.
# No-op when running locally.
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    # 1. Clone wppmpy (provides download_data.py and correct path structure)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/BrainardLab/wppmpy.git"],
        check=True,
    )

    # 2. Clone the paper repo and add to sys.path (for analysis.* imports)
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/fh862/ellipsoids_eLife2025.git",
        ],
        check=True,
    )
    sys.path.insert(0, "/content/ellipsoids_eLife2025")

    # 3. Install paper repo dependencies (jax, scipy, pandas, shapely, ...)
    subprocess.run(
        [
            "pip",
            "install",
            "-q",
            "git+https://github.com/fh862/ellipsoids_eLife2025.git",
        ],
        check=True,
    )

    # 4. Download the required OSF data subset
    subprocess.run(
        [
            "python",
            "/content/wppmpy/src/hong_etal_2025/download_data.py",
            "--data-dir",
            "/content/wppmpy/data",
        ],
        check=True,
    )

    # 5. Set CWD so pathlib.Path("../...") resolves to the wppmpy repo root
    os.chdir("/content/wppmpy/src/hong_etal_2025")

# Threshold Ellipses from Pre-computed Tables

Reproduces **Figure 2C** of Hong et al. (2025) — a 7 × 7 grid of threshold ellipses
in the 2-D model (W-space) for the isoluminant colour plane — by reading the
pre-computed covariance matrices from the OSF dataset and calling into the paper
repository for colour mapping.

No model fitting or JAX computation is required; the threshold covariance matrices
are read directly from the CSV tables provided on OSF.

## Subject mapping (sub number → initials)

| sub | initials |
|-----|----------|
| 1   | CH       |
| 2   | ME       |
| 4   | SG       |
| 6   | DK       |
| 7   | BH       |
| 8   | FM       |
| 10  | HG       |
| 11  | FW       |

## Before running

Download the required data files once:

```bash
python src/hong_etal_2025/download_data.py
```

Install the paper repository into this environment (if not already done):

```bash
pip install -e /path/to/ellipsoids_eLife2025
```

## 1. Setup

In [ ]:
# ruff: noqa: E402
import ast
import pathlib
import warnings

# ── Paper-repo imports ────────────────────────────────────────────────────────
# ellipsoids_eLife2025 must be installed:  pip install -e /path/to/ellipsoids_eLife2025
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

jax.config.update("jax_enable_x64", True)  # required by the paper repo

from analysis.color_thres import color_thresholds

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = pathlib.Path("../..")
DATA_DIR = REPO_ROOT / "data"

# Subject to plot
SUB_N = 1  # 1=CH, 2=ME, 4=SG, 6=DK, 7=BH, 8=FM, 10=HG, 11=FW
SUB_INITIALS = {
    1: "CH",
    2: "ME",
    4: "SG",
    6: "DK",
    7: "BH",
    8: "FM",
    10: "HG",
    11: "FW",
}
PLANE = "Isoluminant plane"

## 2. Load threshold-ellipse covariance matrices

The file `Thres_ellipses_sub{N}.csv` contains one row per reference location on the
7 × 7 grid.  Each row stores:

- **`grid_ref`** — the 2-D W-space coordinates of the reference stimulus,
  as a comma-separated string `"x,y"`.
- **`Sigmas_thres_grid_org`** — the 2 × 2 threshold covariance matrix
  (the boundary of this ellipse is the 66.7 % correct discrimination contour),
  serialised as `"[[s11,s12],[s21,s22]]"`.  Bootstrap replicates occupy the
  remaining columns (not used here).

In [ ]:
thres_csv = (
    DATA_DIR
    / "Organized data and model predictions"
    / f"sub{SUB_N}"
    / f"Thres_ellipses_sub{SUB_N}.csv"
)

df = pd.read_csv(thres_csv)
print(f"Loaded {len(df)} rows from {thres_csv.name}")

# Parse grid reference coordinates: "x,y" → shape (49, 2)
grid_flat = np.array(
    [list(map(float, s.split(","))) for s in df["grid_ref"]]
)  # (49, 2)

# Parse threshold covariance matrices: "[[s11,s12],[s21,s22]]" → shape (49, 2, 2)
sigmas_flat = np.array(
    [ast.literal_eval(s) for s in df["Sigmas_thres_grid_org"]]
)  # (49, 2, 2)

# Reshape into 7 × 7 grids
num_pts = int(round(len(grid_flat) ** 0.5))  # 7
grid = grid_flat.reshape(num_pts, num_pts, 2)  # (7, 7, 2)
Sigmas_thres = sigmas_flat.reshape(num_pts, num_pts, 2, 2)  # (7, 7, 2, 2)

print(f"Grid shape:   {grid.shape}")
print(f"Sigmas shape: {Sigmas_thres.shape}")
print(f"Grid extents: {grid[:, :, 0].min():.2f} – {grid[:, :, 0].max():.2f}  (dim 1)")
print(f"              {grid[:, :, 1].min():.2f} – {grid[:, :, 1].max():.2f}  (dim 2)")

## 3. Colour mapping

Each ellipse is coloured to approximately match the appearance of its reference
stimulus.  `color_thresholds.W2D_to_rgb()` converts a 2-D W-space coordinate to a
display RGB triplet via the monitor calibration matrices.

In [ ]:
# Instantiate and load the colour-transformation object.
# _find_exact_path walks DATA_DIR recursively, so the files in
# data/Calibration and transformation/Transformation matrices/  are found automatically.
color_thres = color_thresholds(2, str(DATA_DIR.resolve()), PLANE)

with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore"
    )  # suppress expected out-of-gamut warnings at grid edges
    color_thres.load_transformation_matrix(file_date="02242025")

print("Transformation matrices loaded.")

# Quick sanity check: convert the centre point (should be close to the display white)
rgb_centre = color_thres.W2D_to_rgb(np.array([0.0, 0.0])).squeeze()
print(f"RGB at W-space origin: {rgb_centre.round(3)}")

## 4. Reconstruct ellipse outlines

Each 2 × 2 threshold covariance matrix $\boldsymbol{\Sigma}$ defines an ellipse:

$$\mathcal{E} = \{\mathbf{c} + \mathbf{L}\mathbf{u} \;:\; \|\mathbf{u}\| = 1\}$$

where $\mathbf{L}$ is the Cholesky factor ($\mathbf{L}\mathbf{L}^\top = \boldsymbol{\Sigma}$)
and $\mathbf{c}$ is the reference location.  This traces the boundary of the set
$\{\mathbf{x} : (\mathbf{x}-\mathbf{c})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\mathbf{c}) \leq 1\}$,
i.e. the 66.7 % correct discrimination contour.

In [ ]:
N_THETA = 200  # points per ellipse
theta = np.linspace(0, 2 * np.pi, N_THETA)
unit_circle = np.vstack([np.cos(theta), np.sin(theta)])  # (2, N_THETA)

# Pre-compute (ellipse_xy, color) for every grid point
ellipses = []  # list of (xy_array, rgb)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress out-of-gamut warnings at grid edges
    for idx in np.ndindex(num_pts, num_pts):
        center = grid[idx]  # (2,)
        Sigma = Sigmas_thres[idx]  # (2, 2)
        L = np.linalg.cholesky(Sigma)  # lower-triangular Cholesky factor
        xy = center[:, None] + L @ unit_circle  # (2, N_THETA)
        rgb = color_thres.W2D_to_rgb(center).squeeze()  # (3,)
        ellipses.append((xy, rgb))

print(f"Computed {len(ellipses)} ellipses.")

## 5. Figure 2C — threshold ellipses on the 7 × 7 grid

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), dpi=150)

for xy, rgb in ellipses:
    ax.plot(xy[0], xy[1], color=rgb, lw=1.2)

# Axis formatting to match the paper
ticks = np.linspace(-0.7, 0.7, 5)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.set_xlabel("W dim 1", fontsize=9)
ax.set_ylabel("W dim 2", fontsize=9)
ax.set_title(
    f"Threshold ellipses — sub{SUB_N} ({SUB_INITIALS[SUB_N]}) — {PLANE}",
    fontsize=10,
)

plt.tight_layout()
plt.show()

## 6. Bootstrap confidence regions

### 6a. Parse bootstrap covariance matrices

In [ ]:
# Bootstrap columns are all columns except 'grid_ref' and 'Sigmas_thres_grid_org'
btst_cols = [c for c in df.columns if c.startswith("Sigmas_thres_grid_btst")]
N_BTST = len(btst_cols)
print(f"Number of bootstrap datasets: {N_BTST}")

# Parse: shape (49, N_BTST, 2, 2)
sigmas_btst_flat = np.array(
    [[ast.literal_eval(df[col].iloc[i]) for col in btst_cols] for i in range(len(df))]
)  # (49, 120, 2, 2)

# Reshape to (7, 7, N_BTST, 2, 2)
Sigmas_btst = sigmas_btst_flat.reshape(num_pts, num_pts, N_BTST, 2, 2)
print(f"Bootstrap Sigmas shape: {Sigmas_btst.shape}")

### 6b. Rank bootstraps by Normalized Bures Similarity (NBS) and retain top 95 %

For each bootstrap dataset we sum the NBS score between the bootstrap Sigma and the
original Sigma across all 49 grid points.  The top 95 % (114 of 120) are retained,
following the method in Hong et al. (2025).

In [ ]:
import numpy.typing as npt
from scipy.linalg import sqrtm as scipy_sqrtm


def nbs(M1: npt.NDArray[np.float64], M2: npt.NDArray[np.float64]) -> float:
    """Normalized Bures Similarity between two 2x2 PD matrices."""
    inner = scipy_sqrtm(scipy_sqrtm(M1) @ M2 @ scipy_sqrtm(M1))
    trace_val = np.real(np.trace(inner))
    norm = np.sqrt(np.trace(M1) * np.trace(M2))
    return float(trace_val / norm)


CI_FRACTION = 0.95
N_KEEP = max(1, int(N_BTST * CI_FRACTION))  # 114
print(f"Retaining top {N_KEEP} of {N_BTST} bootstrap datasets")

# Sum NBS over all 49 grid points for each bootstrap
nbs_sum = np.zeros(N_BTST)
for b in range(N_BTST):
    for idx in np.ndindex(num_pts, num_pts):
        nbs_sum[b] += nbs(Sigmas_thres[idx], Sigmas_btst[idx][b])

# Sort descending, keep top N_KEEP
idx_sorted = np.argsort(nbs_sum)[::-1]
idx_keep = idx_sorted[:N_KEEP]
Sigmas_btst_kept = Sigmas_btst[:, :, idx_keep, :, :]  # (7, 7, 114, 2, 2)
kept_min = nbs_sum[idx_keep].min()
kept_max = nbs_sum[idx_keep].max()
print(f"NBS sum range (kept): {kept_min:.4f} - {kept_max:.4f}")

### 6c. Convert covariance matrices to ellipse parameters

Each 2 × 2 Sigma is converted to `[cx, cy, a, b, theta_rad]` via eigendecomposition,
ready for `find_inner_outer_contours` from the paper repo.

In [ ]:
from typing import Any

import numpy.typing as npt
from analysis.ellipses_tools import covMat_to_ellParamsQ


def sigma_to_params(
    center: npt.NDArray[np.float64], Sigma: npt.NDArray[np.float64]
) -> list[Any]:
    """Return [cx, cy, a, b, theta_rad] from a centre and 2x2 covariance matrix."""
    _, _, axes_lengths, theta_deg = covMat_to_ellParamsQ(Sigma)
    return [
        center[0],
        center[1],
        axes_lengths[0],
        axes_lengths[1],
        theta_deg,
    ]


# Original ellipse params: shape (7, 7, 5)
grid_indices = list(np.ndindex(num_pts, num_pts))
params_org = np.array(
    [sigma_to_params(grid[idx], Sigmas_thres[idx]) for idx in grid_indices]
).reshape(num_pts, num_pts, 5)

# Bootstrap ellipse params: shape (7, 7, N_KEEP, 5)
params_btst = np.full((num_pts, num_pts, N_KEEP, 5), np.nan)
for idx in grid_indices:
    for k in range(N_KEEP):
        params_btst[idx][k] = sigma_to_params(grid[idx], Sigmas_btst_kept[idx][k])

print(f"params_org shape:   {params_org.shape}")
print(f"params_btst shape:  {params_btst.shape}")

In [ ]:
# For each angle theta, the radial distance from the ellipse centre to the
# boundary of the ellipse defined by Sigma is:
#   r(theta) = 1 / sqrt( v^T Sigma^{-1} v )   where v = [cos(theta), sin(theta)]
#
# We compute r for every retained bootstrap Sigma and every angle, then
# take the min and max across bootstraps to obtain the inner and outer
# confidence envelopes.  No shapely required.

N_THETA_CI = 360
theta_ci = np.linspace(0, 2 * np.pi, N_THETA_CI, endpoint=False)
dirs = np.column_stack([np.cos(theta_ci), np.sin(theta_ci)])  # (N_THETA_CI, 2)

# r_btst[i, j, k, t] = radial distance for bootstrap k at angle t, grid point (i,j)
r_btst = np.full((num_pts, num_pts, N_KEEP, N_THETA_CI), np.nan)
for idx in np.ndindex(num_pts, num_pts):
    for k in range(N_KEEP):
        S = Sigmas_btst_kept[idx][k]
        S_inv = np.linalg.inv(S)
        # v^T S^{-1} v for each direction
        quad = np.einsum("td,de,te->t", dirs, S_inv, dirs)  # (N_THETA_CI,)
        r_btst[idx][k] = 1.0 / np.sqrt(quad)

# CI envelopes: min and max radial distance across bootstraps
r_outer = r_btst.max(axis=2)  # (7, 7, N_THETA_CI)
r_inner = r_btst.min(axis=2)  # (7, 7, N_THETA_CI)

print(f"r_outer shape: {r_outer.shape}")
# Quick sanity check at corner (0, 6)
i, j = 0, 6
S = Sigmas_thres[i, j]
S_inv = np.linalg.inv(S)
r_org = 1.0 / np.sqrt(np.einsum("td,de,te->t", dirs, S_inv, dirs))
print(f"Original r range at (0,6): {r_org.min():.4f} – {r_org.max():.4f}")
print(
    f"Outer    r range at (0,6): {r_outer[i, j].min():.4f} – {r_outer[i, j].max():.4f}"
)
print(
    f"Inner    r range at (0,6): {r_inner[i, j].min():.4f} – {r_inner[i, j].max():.4f}"
)

## 7. Figure 2C with 95 % bootstrap confidence regions

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), dpi=150)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for idx in np.ndindex(num_pts, num_pts):
        center = grid[idx]
        rgb = color_thres.W2D_to_rgb(center).squeeze()

        # Confidence band: fill between outer and inner radial envelopes
        outer_xy = center[:, None] + r_outer[idx] * dirs.T  # (2, N_THETA_CI)
        inner_xy = center[:, None] + r_inner[idx] * dirs.T  # (2, N_THETA_CI)

        # Build band polygon: outer contour forward, inner contour backward
        band_x = np.concatenate([outer_xy[0], inner_xy[0, ::-1], [outer_xy[0, 0]]])
        band_y = np.concatenate([outer_xy[1], inner_xy[1, ::-1], [outer_xy[1, 0]]])
        ax.fill(band_x, band_y, color=rgb, alpha=0.4, linewidth=0)

        # Main ellipse in black (Cholesky)
        theta_plot = np.linspace(0, 2 * np.pi, 200)
        uc = np.vstack([np.cos(theta_plot), np.sin(theta_plot)])
        L = np.linalg.cholesky(Sigmas_thres[idx])
        xy = center[:, None] + L @ uc
        ax.plot(xy[0], xy[1], color="black", lw=0.8)

ticks = np.linspace(-0.7, 0.7, 5)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.set_xlabel("W dim 1", fontsize=9)
ax.set_ylabel("W dim 2", fontsize=9)
ax.set_title(
    f"Threshold ellipses + 95% CI — sub{SUB_N} ({SUB_INITIALS[SUB_N]}) — {PLANE}",
    fontsize=9,
)
plt.tight_layout()
plt.show()